# 章节实践：基于 Blaze 的 MXFP8 量化矩阵算子

通过本章的系统学习，我们已经掌握了 Blaze 模板体系下 MXFP8 量化 MatMul 的分层架构与开发流程。现提供以下**循序渐进**的实践练习：基于 Blaze 实现 **QuantMatmulMxfp8Swat** 量化矩阵乘算子，先补齐 Host Tiling，验证 TilingData 输出后再逐步完成 Kernel、Block 层代码。

算子 I/O 规格如下：

<div style="width: 100%;">
<table style="border-collapse: collapse; width: 650px; max-width: 650px; border: 1px solid #bbb; margin-left: 0 !important; margin-right: auto; text-align: left;">
  <tr>
    <td colspan="1" style="padding: 8px; border: 1px solid #bbb; font-weight: bold;">算子类型</td>
    <td colspan="4" style="padding: 8px; border: 1px solid #bbb; font-weight: bold; text-align: center;">QuantMatmulMxfp8Swat</td>
  </tr>
  <tr>
    <td rowspan="5" style="padding: 8px; border: 1px solid #bbb; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输入</td>
    <td style="padding: 8px; border: 1px solid #bbb; font-weight: bold;">name</td>
    <td style="padding: 8px; border: 1px solid #bbb; font-weight: bold;">shape</td>
    <td style="padding: 8px; border: 1px solid #bbb; font-weight: bold;">data type</td>
    <td style="padding: 8px; border: 1px solid #bbb; font-weight: bold;">format</td>
  </tr>
  <tr>
    <td style="padding: 8px; border: 1px solid #bbb;">a</td>
    <td style="padding: 8px; border: 1px solid #bbb;">(m, k)</td>
    <td style="padding: 8px; border: 1px solid #bbb;">float8_e4m3</td>
    <td style="padding: 8px; border: 1px solid #bbb;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border: 1px solid #bbb;">b</td>
    <td style="padding: 8px; border: 1px solid #bbb;">(n, k)</td>
    <td style="padding: 8px; border: 1px solid #bbb;">float8_e4m3</td>
    <td style="padding: 8px; border: 1px solid #bbb;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border: 1px solid #bbb;">scale_a</td>
    <td style="padding: 8px; border: 1px solid #bbb;">(m, ceil(k/64), 2)</td>
    <td style="padding: 8px; border: 1px solid #bbb;">float8_e8m0</td>
    <td style="padding: 8px; border: 1px solid #bbb;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border: 1px solid #bbb;">scale_b</td>
    <td style="padding: 8px; border: 1px solid #bbb;">(n, ceil(k/64), 2)</td>
    <td style="padding: 8px; border: 1px solid #bbb;">float8_e8m0</td>
    <td style="padding: 8px; border: 1px solid #bbb;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border: 1px solid #bbb; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输出</td>
    <td style="padding: 8px; border: 1px solid #bbb;">c</td>
    <td style="padding: 8px; border: 1px solid #bbb;">(m, n)</td>
    <td style="padding: 8px; border: 1px solid #bbb;">bfloat16</td>
    <td style="padding: 8px; border: 1px solid #bbb;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border: 1px solid #bbb; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">量化模式</td>
    <td colspan="4" style="padding: 8px; border: 1px solid #bbb;">G-G 量化（group size = 32）</td>
  </tr>
  <tr>
    <td style="padding: 8px; border: 1px solid #bbb; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">测试规模</td>
    <td colspan="4" style="padding: 8px; border: 1px solid #bbb;">m=16, k=128, n=16384</td>
  </tr>
  <tr>
    <td style="padding: 8px; border: 1px solid #bbb; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">layout</td>
    <td colspan="4" style="padding: 8px; border: 1px solid #bbb;">A 非转置（transA=0），B 转置（transB=1）</td>
  </tr>
  <tr>
    <td style="padding: 8px; border: 1px solid #bbb; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">目标硬件</td>
    <td colspan="4" style="padding: 8px; border: 1px solid #bbb;">Ascend950 PR/DT（NPU ARCH 3510）</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>

**练习步骤：**

1. **Host Tiling 补齐**：完成 `DoOpTiling`、`CalcScaleKL1`、`BuildTilingData` 中标记 `TODO` 的部分；对照参考答案自行修正后，运行 `--tiling-only` 验证 TilingData 输出。
2. **Kernel 层补齐**：完成 `operator()`、`ResetGmAddr`、`Process` 中标记 `TODO` 的部分。
3. **Block 层补齐**：完成 `BlockMmad::operator()` 中 MX MMAD 调用处标记 `TODO` 的部分。
4. **端到端测试**：编译运行完整算子，NPU 输出与 CPU golden 一致（`[PASS]`）。

**涉及文件：**

<div style="width: 100%;">
<table style="border-collapse: collapse; width: auto; max-width: 100%; margin-left: 0 !important; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left; padding: 6px 12px 6px 0; border-bottom: 1px solid #ddd;">文件</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">职责</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">include/tiling/quant_matmul_mx_tiling_swat.h</td>
      <td style="text-align: left; padding: 6px 12px;">Host Tiling：分块参数计算与 QuantMatmulTilingData 填充</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">include/kernel/quant_matmul_mx_kernel_swat.h</td>
      <td style="text-align: left; padding: 6px 12px;">Blaze Kernel 层：tile 循环编排与 BlockMmad 驱动</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">include/block/quant_matmul_mx_block_mmad_swat.h</td>
      <td style="text-align: left; padding: 6px 12px;">Blaze Block 层：L1/L0 双缓冲 Load/MMAD 流水线</td>
    </tr>
  </tbody>
</table>
</div>

请开始你的实践，体验从理解到创造的完整开发过程。

---

## 环境准备：

In [ ]:
import os, subprocess

env = subprocess.check_output(
    "bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'",
    shell=True, text=True
)
for line in env.splitlines():
    if "=" in line:
        os.environ.__setitem__(*line.split("=", 1))

print("Environment initialized.")
print(f"ASCEND_TOOLKIT_HOME={os.environ.get('ASCEND_TOOLKIT_HOME', 'N/A')}")

In [ ]:
# 检查 NPU 设备
!npu-smi info 2>/dev/null || echo "[提示] 非 Ascend950 PR/DT 环境，实操编译运行请在支持硬件平台上完成"

---

## 工程结构

教程工程位于 `src/01.07_blaze_based_matmul_operator_development/`。Launcher 与 Tile/Policy 层已提供完整实现，Host Tiling、Kernel、Block 三层需按步骤补齐标记 `TODO` 的代码。

以下命令可查看工程目录：

In [ ]:
!tree src/01.07_blaze_based_matmul_operator_development/ -L 2 2>/dev/null || find src/01.07_blaze_based_matmul_operator_development/ -maxdepth 2 -print | sed 's|[^/]*/| |g'

---

## 步骤 1：Host Tiling 代码补齐

修改以下代码，完成 `DoOpTiling`、`CalcScaleKL1`、`BuildTilingData` 三个函数中标记 `TODO` 的部分。

In [ ]:
%%writefile src/01.07_blaze_based_matmul_operator_development/include/tiling/quant_matmul_mx_tiling_swat.h

/**
 * Copyright (c) 2026 Huawei Technologies Co., Ltd.
 * This program is free software, you can redistribute it and/or modify it under the terms and conditions of
 * CANN Open Software License Agreement Version 2.0 (the "License").
 * Please refer to the License for details. You may not use this file except in compliance with the License.
 * THIS SOFTWARE IS PROVIDED ON AN "AS IS" BASIS, WITHOUT WARRANTIES OF ANY KIND, EITHER EXPRESS OR IMPLIED,
 * INCLUDING BUT NOT LIMITED TO NON-INFRINGEMENT, MERCHANTABILITY, OR FITNESS FOR A PARTICULAR PURPOSE.
 * See LICENSE in the root of the software repository for the full text of the License.
 */

/*!
 * \file quant_matmul_mx_tiling_swat.h
 * \brief SWAT tiling specialization for the MX non-full-load path.
 */

#pragma once

#include <algorithm>

#include "quant_matmul_tiling_base.h"

template <mm::DataType aDataType, mm::DataType bDataType>
class QuantMatmulTilingSwat : public QuantMatmulTilingBase<aDataType, bDataType> {
public:
    QuantMatmulTilingSwat() = default;
    ~QuantMatmulTilingSwat() override = default;

private:
    using Base = QuantMatmulTilingBase<aDataType, bDataType>;
    using Base::args_;
    using Base::platformInfo_;
    using Base::runInfo_;

protected:
    const char* TilingName() const override
    {
        return "swat";
    }

    void DoOpTiling(QuantMatmulTilingData& tilingData) override
    {
        // TODO: 依次调用分块搜索、尾块优化、L1 深度计算等 Host Tiling 主流程
        // 提示: CalcBasicBlock(); OptimizeEdgeBasicBlock(); CalcTailBasicBlock(); CalcPathSpecificL1();
        // TODO: 计算 scaleKL1，并调用 BuildTilingData 填充 QuantMatmulTilingData
    }

private:
    uint32_t CalcScaleKL1() const
    {
        // TODO: 返回 A/B 两侧 Scale 复用 K 窗口的较小值
        // 提示: min(scaleFactorA * stepKa * baseK, scaleFactorB * stepKb * baseK)
        return 0;
    }

    void BuildTilingData(QuantMatmulTilingData& tilingData, uint32_t scaleKL1, uint8_t nBufferNum) const
    {
        // TODO: 将 runInfo_ / args_ 中的分块结果写入 QuantMatmulTilingData
        // 提示: 需填充 m/n/k, baseM/N/K, mTailTile/nTailTile, usedCoreNum, scaleKL1, stepK, nBufferNum, dbL0c 等字段
        (void)scaleKL1;
        (void)nBufferNum;
    }

    void CalcTailBasicBlock()
    {
        if (runInfo_.tailBlockCnt == 0UL) {
            return;
        }

        // Non-full-load can split both M and N tail tiles. Grow the heavier
        // edge first, but keep the total tail work within the available cores.
        uint64_t mTile = 1UL;
        uint64_t nTile = 1UL;
        uint64_t preSplit = 1UL;
        uint64_t secSplit = 1UL;
        uint64_t& preSplitValid = runInfo_.mTailSize >= runInfo_.nTailSize ? mTile : nTile;
        uint64_t& secSplitValid = runInfo_.mTailSize >= runInfo_.nTailSize ? nTile : mTile;
        uint64_t tileMax = platformInfo_.aicNum / runInfo_.tailBlockCnt;
        uint64_t mTileMax = std::min(tileMax, CeilDiv(runInfo_.baseM, CUBE_BLOCK));
        uint64_t nTileMax = std::min(tileMax, CeilDiv(runInfo_.baseN, CUBE_BLOCK));
        uint64_t preSplitMax = runInfo_.mTailSize >= runInfo_.nTailSize ? mTileMax : nTileMax;
        uint64_t secSplitMax = runInfo_.mTailSize >= runInfo_.nTailSize ? nTileMax : mTileMax;
        while ((CalUsedCoreNum(runInfo_, preSplit + 1UL, secSplit) <= platformInfo_.aicNum && preSplit < preSplitMax) ||
               (CalUsedCoreNum(runInfo_, preSplit, secSplit + 1UL) <= platformInfo_.aicNum && secSplit < secSplitMax)) {
            if (CalUsedCoreNum(runInfo_, preSplit + 1UL, secSplit) <= platformInfo_.aicNum && preSplit < preSplitMax) {
                preSplitValid = ++preSplit;
            }
            if (CalUsedCoreNum(runInfo_, preSplit, secSplit + 1UL) <= platformInfo_.aicNum && secSplit < secSplitMax) {
                secSplitValid = ++secSplit;
            }
        }

        runInfo_.mTailTile = mTile;
        runInfo_.nTailTile = nTile;
    }

    void CalcPathSpecificL1()
    {
        // This path allocates A, B, scaleA, and scaleB symmetrically, so it
        // first searches a shared depth and then refines A/B independently.
        uint64_t baseASize = GetSizeWithDataType<aDataType>(runInfo_.baseM * runInfo_.baseK);
        uint64_t baseBSize = GetSizeWithDataType<bDataType>(runInfo_.baseN * runInfo_.baseK);

        uint64_t baseScaleASize =
            Align(CeilDiv(runInfo_.baseK, MX_GROUP_SIZE), TILING_MXFP_MULTI_BASE_SIZE) * runInfo_.baseM;
        uint64_t baseScaleBSize =
            Align(CeilDiv(runInfo_.baseK, MX_GROUP_SIZE), TILING_MXFP_MULTI_BASE_SIZE) * runInfo_.baseN;
        uint64_t baseL1Size = baseASize + baseBSize + baseScaleASize + baseScaleBSize;
        uint64_t depthInit = GetDepthA1B1(runInfo_, platformInfo_.l1Size, baseL1Size, 1UL);
        uint64_t leftL1SizeByDepthInit = platformInfo_.l1Size - depthInit * baseL1Size;
        uint64_t depthASec =
            GetDepthA1B1(runInfo_, leftL1SizeByDepthInit, (baseASize + baseScaleASize) * depthInit, depthInit);
        uint64_t depthBSec =
            GetDepthA1B1(runInfo_, leftL1SizeByDepthInit, (baseBSize + baseScaleBSize) * depthInit, depthInit);
        // Start from a symmetric A/B depth and only break symmetry if the
        // budget cannot sustain the larger common depth on both sides.
        runInfo_.depthA1 = std::max(depthASec, depthBSec);
        runInfo_.depthB1 = runInfo_.depthA1;
        if (runInfo_.depthA1 * baseL1Size > platformInfo_.l1Size) {
            runInfo_.depthA1 = depthASec >= depthBSec ? depthASec : depthInit;
            runInfo_.depthB1 = depthASec < depthBSec ? depthBSec : depthInit;
        }
        CalStepKs(args_, runInfo_);
        CalScaleFactors(args_, platformInfo_, runInfo_, baseASize, baseBSize, baseScaleASize, baseScaleBSize);
    }

    void AdjustBasicBlock()
    {
        // Re-balance the initial M/N split when the first guess underutilizes
        // cores or produces a highly skewed tile.
        uint64_t baseMAlignNum = args_.transA ? GetShapeWithDataType<aDataType>(L2_ALIGN_SIZE) : CUBE_BLOCK;
        uint64_t baseNAlignNum = args_.transB ? CUBE_BLOCK : GetShapeWithDataType<bDataType>(L2_ALIGN_SIZE);
        uint64_t baseKAlignNum = (args_.transA && !args_.transB) ? GetShapeWithDataType<aDataType>(FP8_C0_SIZE) :
                                                                   GetShapeWithDataType<aDataType>(L2_ALIGN_SIZE);
        if (args_.transA || !args_.transB) {
            baseKAlignNum = GetShapeWithDataType<aDataType>(MXFP_DIVISOR_SIZE);
        }
        uint64_t mMaxtile = CeilDiv(args_.m, baseMAlignNum);
        uint64_t nMaxtile = CeilDiv(args_.n, baseNAlignNum);
        uint64_t tempBaseM = runInfo_.baseM;
        uint64_t tempBaseN = runInfo_.baseN;
        uint64_t coreNumMN = platformInfo_.aicNum;

        if (mMaxtile * nMaxtile >= coreNumMN || (!args_.transA && args_.transB)) {
            uint64_t mCnt = CeilDiv(args_.m, runInfo_.baseM);
            uint64_t nCnt = CeilDiv(args_.n, runInfo_.baseN);

            if (mMaxtile > nMaxtile) {
                tempBaseN = Align(CeilDiv(args_.n, nCnt), baseNAlignNum);
                nCnt = CeilDiv(args_.n, tempBaseN);
                mCnt = platformInfo_.aicNum / nCnt;
                tempBaseM = Align(CeilDiv(args_.m, mCnt), baseMAlignNum);
            } else {
                tempBaseM = Align(CeilDiv(args_.m, mCnt), baseMAlignNum);
                mCnt = CeilDiv(args_.m, tempBaseM);
                nCnt = platformInfo_.aicNum / mCnt;
                tempBaseN = Align(CeilDiv(args_.n, nCnt), baseNAlignNum);
            }

            while (tempBaseN > tempBaseM * BASEM_BASEN_RATIO && nCnt < platformInfo_.aicNum / NUM_TWO &&
                   tempBaseN != baseNAlignNum) {
                nCnt = nCnt * NUM_TWO;
                mCnt = platformInfo_.aicNum / nCnt;
                tempBaseM = Align(CeilDiv(args_.m, mCnt), baseMAlignNum);
                tempBaseN = Align(CeilDiv(args_.n, nCnt), baseNAlignNum);
                mCnt = CeilDiv(args_.m, tempBaseM);
                nCnt = CeilDiv(args_.n, tempBaseN);
            }
            while (tempBaseM >= tempBaseN * BASEM_BASEN_RATIO && mCnt < platformInfo_.aicNum / NUM_TWO &&
                   tempBaseM != baseMAlignNum) {
                mCnt = mCnt * NUM_TWO;
                nCnt = platformInfo_.aicNum / mCnt;
                tempBaseM = Align(CeilDiv(args_.m, mCnt), baseMAlignNum);
                tempBaseN = Align(CeilDiv(args_.n, nCnt), baseNAlignNum);
                mCnt = CeilDiv(args_.m, tempBaseM);
                nCnt = CeilDiv(args_.n, tempBaseN);
            }

            uint64_t kAlignValue = Align(args_.k, baseKAlignNum);
            uint64_t kMaxValue =
                GetShapeWithDataType<aDataType>(platformInfo_.l0aSize / DB_SIZE) / std::max(tempBaseM, tempBaseN);
            kMaxValue = FloorAlign(kMaxValue, baseKAlignNum);
            if (kMaxValue >= baseKAlignNum) {
                runInfo_.baseM = tempBaseM;
                runInfo_.baseN = tempBaseN;
                runInfo_.baseK = std::min(kAlignValue, kMaxValue);
                runInfo_.baseK = runInfo_.baseK > BASEK_LIMIT ? Align(runInfo_.baseK / NUM_TWO, BASIC_BLOCK_SIZE_256) :
                                                                runInfo_.baseK;
            }
        }
    }

    void CalcBasicBlock()
    {
        // Start from a 256-sized candidate tile, then refine it and capture
        // the tail statistics used by later scheduling decisions.
        runInfo_.baseM = std::min(args_.m, BASIC_BLOCK_SIZE_256);
        runInfo_.baseM = !args_.transA ? Align(runInfo_.baseM, CUBE_BLOCK) :
                                         Align(runInfo_.baseM, GetShapeWithDataType<aDataType>(L1_ALIGN_SIZE));
        runInfo_.baseN = std::min(args_.n, BASIC_BLOCK_SIZE_256);
        runInfo_.baseN = args_.transB ? Align(runInfo_.baseN, CUBE_BLOCK) :
                                        Align(runInfo_.baseN, GetShapeWithDataType<bDataType>(L1_ALIGN_SIZE));
        runInfo_.baseK = Align(
            std::min(args_.k, aDataType == mm::DataType::DT_FLOAT4_E2M1 ? BASIC_BLOCK_SIZE_256 : BASIC_BLOCK_SIZE_128),
            TILING_MXFP_DIVISOR_SIZE);

        uint64_t blockNum = CeilDiv(args_.m, runInfo_.baseM) * CeilDiv(args_.n, runInfo_.baseN);
        if (blockNum < platformInfo_.aicNum) {
            AdjustBasicBlock();
        }
        CHECK_COND(
            runInfo_.baseM != 0UL && runInfo_.baseN != 0UL && runInfo_.baseK != 0UL,
            "Failed to derive a valid tiling base shape: baseM, baseN, and baseK must all be non-zero.");

        runInfo_.mBlockCnt = CeilDiv(args_.m, runInfo_.baseM);
        runInfo_.nBlockCnt = CeilDiv(args_.n, runInfo_.baseN);
        runInfo_.totalBlockCnt = runInfo_.mBlockCnt * runInfo_.nBlockCnt;
        runInfo_.tailBlockCnt = runInfo_.totalBlockCnt % platformInfo_.aicNum;
        runInfo_.mTailSize = args_.m - (runInfo_.mBlockCnt - 1UL) * runInfo_.baseM;
        runInfo_.nTailSize = args_.n - (runInfo_.nBlockCnt - 1UL) * runInfo_.baseN;
        runInfo_.dbL0c =
            runInfo_.baseM * runInfo_.baseN * DATA_SIZE_L0C * DB_SIZE <= platformInfo_.l0cSize ? DB_SIZE : 1U;
    }

    void OptimizeEdgeBasicBlock()
    {
        // Merge tiny edge tiles when K is cache-line aligned so the tail block
        // behaves more like the steady-state region on both M/N directions.
        if (runInfo_.mBlockCnt == 1UL && runInfo_.nBlockCnt == 1UL) {
            return;
        }

        bool isInnerAxisAlign = GetSizeWithDataType<aDataType>(args_.k) % MTE2_CACHELINE_SIZE == 0UL;

        uint64_t mTailSize = args_.m % runInfo_.baseM;
        if (runInfo_.mBlockCnt > 1UL && mTailSize > 0UL && !args_.transA && isInnerAxisAlign) {
            uint64_t baseTailCntMax = std::min((runInfo_.baseM - mTailSize) / BASIC_BLOCK_SIZE_16, runInfo_.mBlockCnt);
            uint64_t windowSize = std::min(WINDOW_LEN, runInfo_.mBlockCnt);
            uint64_t mainWindowNum = runInfo_.mBlockCnt / windowSize - 1UL;
            uint64_t tailWindowSize = runInfo_.mBlockCnt - mainWindowNum * windowSize;
            uint64_t perfRes = (mainWindowNum + 1UL) * runInfo_.baseM;
            uint64_t mergeWindowNum = 1UL;

            for (uint64_t mergeLen = tailWindowSize - 1UL; mergeLen < baseTailCntMax;
                 mergeLen += windowSize, ++mergeWindowNum) {
                uint64_t newTailMain =
                    Align(CeilDiv((mergeLen * runInfo_.baseM + mTailSize), mergeLen + 1UL), BASIC_BLOCK_SIZE_16);
                uint64_t curPerf =
                    (mainWindowNum + 1UL - mergeWindowNum) * runInfo_.baseM + mergeWindowNum * newTailMain;
                if (curPerf <= perfRes) {
                    perfRes = curPerf;
                    runInfo_.mTailMain = newTailMain;
                    runInfo_.mBaseTailSplitCnt = mergeLen + 1UL;
                }
            }
        }

        uint64_t nTailSize = args_.n % runInfo_.baseN;
        if (runInfo_.nBlockCnt > 1UL && nTailSize > 0UL && args_.transB && isInnerAxisAlign) {
            uint64_t baseTailCntMax = std::min((runInfo_.baseN - nTailSize) / BASIC_BLOCK_SIZE_16, runInfo_.nBlockCnt);
            uint64_t windowSize = std::min(WINDOW_LEN, runInfo_.nBlockCnt);
            uint64_t mainWindowNum = runInfo_.nBlockCnt / windowSize - 1UL;
            uint64_t tailWindowSize = runInfo_.nBlockCnt - mainWindowNum * windowSize;
            uint64_t perfRes = (mainWindowNum + 1UL) * runInfo_.baseN;
            uint64_t mergeWindowNum = 1UL;

            for (uint64_t mergeLen = tailWindowSize - 1UL; mergeLen < baseTailCntMax;
                 mergeLen += windowSize, ++mergeWindowNum) {
                uint64_t newTailMain =
                    Align(CeilDiv((mergeLen * runInfo_.baseN + nTailSize), mergeLen + 1UL), BASIC_BLOCK_SIZE_16);
                uint64_t curPerf =
                    (mainWindowNum + 1UL - mergeWindowNum) * runInfo_.baseN + mergeWindowNum * newTailMain;
                if (curPerf <= perfRes) {
                    perfRes = curPerf;
                    runInfo_.nTailMain = newTailMain;
                    runInfo_.nBaseTailSplitCnt = mergeLen + 1UL;
                }
            }
        }
    }

    uint64_t CalUsedCoreNum(const QuantMatmulRunInfo& runInfo, uint64_t mTile, uint64_t nTile)
    {
        return mTile * nTile * runInfo.tailBlockCnt;
    }

    uint64_t GetDepthA1B1(
        const QuantMatmulRunInfo& runInfo, uint64_t leftSize, uint64_t perDepthSize, uint64_t depthInit)
    {
        // The first pass grows by powers of two to find a feasible region; the
        // second pass snaps the result to a DMA-friendly K granularity.
        if (depthInit > 1UL && perDepthSize > DB_SIZE * MTE2_MIN_LOAD_SIZE) {
            return depthInit;
        }
        uint64_t depthScale = leftSize / perDepthSize;
        if (depthInit > 1UL) {
            uint64_t baseKSize = GetSizeWithDataType<aDataType>(runInfo.baseK);
            while ((depthScale * baseKSize) % BASIC_BLOCK_SIZE_512 != 0UL &&
                   (depthScale * baseKSize) > BASIC_BLOCK_SIZE_512) {
                depthScale -= 1UL;
            }
            if ((depthScale * baseKSize) % BASIC_BLOCK_SIZE_512 != 0UL &&
                (depthScale * baseKSize) >= BASIC_BLOCK_SIZE_256) {
                depthScale = BASIC_BLOCK_SIZE_256 / baseKSize;
            }
            depthScale = std::max(depthScale, 1UL);
        } else {
            constexpr uint64_t SCALE_INDEX = 2UL;
            depthScale = 1UL;
            while (depthScale * perDepthSize < leftSize) {
                depthScale *= SCALE_INDEX;
            }
            depthScale = depthScale == 1UL ? depthScale : depthScale / SCALE_INDEX;
        }
        return depthInit * depthScale;
    }

    void CalStepKs(const QuantMatmulArgs& args, QuantMatmulRunInfo& runInfo)
    {
        // Convert L1 depth to step-K counts and keep A/B synchronized so both
        // sides advance through K with the same outer scheduling cadence.
        runInfo.stepKa = runInfo.depthA1 / DB_SIZE;
        runInfo.stepKb = runInfo.depthB1 / DB_SIZE;

        if (runInfo.stepKa * runInfo.baseK > args.k) {
            runInfo.stepKa = CeilDiv(args.k, runInfo.baseK);
        }

        if (runInfo.stepKb * runInfo.baseK > args.k) {
            runInfo.stepKb = CeilDiv(args.k, runInfo.baseK);
        }

        if (runInfo.stepKa > runInfo.stepKb) {
            runInfo.stepKa = runInfo.stepKa / runInfo.stepKb * runInfo.stepKb;
        }
        if (runInfo.stepKb > runInfo.stepKa) {
            runInfo.stepKb = runInfo.stepKb / runInfo.stepKa * runInfo.stepKa;
        }

        runInfo.stepKa = std::min(runInfo.stepKa, 4UL);
        runInfo.stepKb = std::min(runInfo.stepKb, 4UL);

        runInfo.depthA1 = runInfo.stepKa * DB_SIZE;
        runInfo.depthB1 = runInfo.stepKb * DB_SIZE;
    }

    void CalScaleFactors(
        const QuantMatmulArgs& args, const QuantMatmulPlatformInfo& platformInfo, QuantMatmulRunInfo& runInfo,
        uint64_t baseASize, uint64_t baseBSize, uint64_t baseScaleASize, uint64_t baseScaleBSize)
    {
        // Scale reuse is solved after A/B depth is fixed. The search keeps the
        // two scale paths balanced while staying inside the leftover L1 budget.
        uint64_t scaleFactorAMax = std::min(MTE2_MIN_LOAD_SIZE / baseScaleASize, SCALER_FACTOR_MAX);
        uint64_t scaleFactorBMax = std::min(MTE2_MIN_LOAD_SIZE / baseScaleBSize, SCALER_FACTOR_MAX);
        uint64_t scaleFactorA = args.k / (runInfo.stepKa * runInfo.baseK);
        uint64_t scaleFactorB = args.k / (runInfo.stepKb * runInfo.baseK);
        runInfo.scaleFactorA = std::max(SCALER_FACTOR_MIN, scaleFactorA);
        runInfo.scaleFactorB = std::max(SCALER_FACTOR_MIN, scaleFactorB);
        runInfo.scaleFactorA = std::min(scaleFactorAMax, runInfo.scaleFactorA);
        runInfo.scaleFactorB = std::min(scaleFactorBMax, runInfo.scaleFactorB);

        // `scaleInit` is the balanced reuse factor both sides can afford
        // without favoring either A or B. Any leftover space is then assigned
        // to the side that can still benefit from deeper scale reuse.
        uint64_t leftL1Size = platformInfo.l1Size - (runInfo.depthA1 * baseASize + runInfo.depthB1 * baseBSize);
        uint64_t scaleInit = leftL1Size / (runInfo.depthA1 * baseScaleASize + runInfo.depthB1 * baseScaleBSize);
        if (runInfo.scaleFactorA <= scaleInit && runInfo.scaleFactorB > scaleInit) {
            leftL1Size -= runInfo.scaleFactorA * runInfo.depthA1 * baseScaleASize;
            runInfo.scaleFactorB = std::min(leftL1Size / (runInfo.depthB1 * baseScaleBSize), runInfo.scaleFactorB);
        } else if (runInfo.scaleFactorB <= scaleInit && runInfo.scaleFactorA > scaleInit) {
            leftL1Size -= runInfo.scaleFactorB * runInfo.depthB1 * baseScaleBSize;
            runInfo.scaleFactorA = std::min(leftL1Size / (runInfo.depthA1 * baseScaleASize), runInfo.scaleFactorA);
        } else if (runInfo.scaleFactorA > scaleInit && runInfo.scaleFactorB > scaleInit) {
            leftL1Size -= scaleInit * runInfo.depthB1 * baseScaleBSize + scaleInit * runInfo.depthA1 * baseScaleASize;
            uint64_t scaleASec =
                std::min(leftL1Size / (runInfo.depthA1 * baseScaleASize), runInfo.scaleFactorA - scaleInit);
            uint64_t scaleBSec =
                std::min(leftL1Size / (runInfo.depthB1 * baseScaleBSize), runInfo.scaleFactorB - scaleInit);
            runInfo.scaleFactorA = scaleASec >= scaleBSec ? scaleASec + scaleInit : scaleInit;
            runInfo.scaleFactorB = scaleASec < scaleBSec ? scaleBSec + scaleInit : scaleInit;
        }
    }
};


---

### Host Tiling 参考答案

请先自行尝试补齐上述代码。完成后对照以下参考答案，**将答案复制到上方 cell 中并执行写入**，再运行 `--tiling-only` 验证 TilingData 是否正确。


In [ ]:
!cat answer/01.07_answer/include/tiling/quant_matmul_mx_tiling_swat.h


---

### 验证 Host Tiling 输出

编译工程并运行 `--tiling-only`，确认 Host 侧分块参数打印正确。

预期输出包含：`baseM=16, baseN=256, baseK=128, scaleKL1=128, usedCoreNum=32`。

In [ ]:
!bash src/01.07_blaze_based_matmul_operator_development/build.sh

In [ ]:
%%bash
SRC="src/01.07_blaze_based_matmul_operator_development"
cd "$SRC/build_out" && ./quant_matmul_mxfp8_tutorial --tiling-only 16 128 16384 0 1

---

## 步骤 2：Kernel 代码补齐

确认 TilingData 输出正确后，修改以下代码，完成 Kernel 层 `operator()`、`ResetGmAddr` 以及 `Process` 循环体中标记 `TODO` 的部分。

In [ ]:
%%writefile src/01.07_blaze_based_matmul_operator_development/include/kernel/quant_matmul_mx_kernel_swat.h

/**
 * Copyright (c) 2026 Huawei Technologies Co., Ltd.
 * This program is free software, you can redistribute it and/or modify it under the terms and conditions of
 * CANN Open Software License Agreement Version 2.0 (the "License").
 * Please refer to the License for details. You may not use this file except in compliance with the License.
 * THIS SOFTWARE IS PROVIDED ON AN "AS IS" BASIS, WITHOUT WARRANTIES OF ANY KIND, EITHER EXPRESS OR IMPLIED,
 * INCLUDING BUT NOT LIMITED TO NON-INFRINGEMENT, MERCHANTABILITY, OR FITNESS FOR A PARTICULAR PURPOSE.
 * See LICENSE in the root of the software repository for the full text of the License.
 */

/*!
 * \file quant_matmul_mx_kernel_swat.h
 * \brief Kernel-side SWAT MX implementation for the non-full-load path.
 */

#pragma once

#if ASC_DEVKIT_MAJOR >= 9
#include "kernel_basic_intf.h"
#else
#include "kernel_operator.h"
#include "kernel_operator_intf.h"
#endif

#include "kernel_utils/common_utils.h"
#include "include/tensor_api/tensor.h"

#include "block/quant_matmul_mx_block_mmad_swat.h"
#include "block/quant_matmul_mx_block_scheduler_swat.h"
#include "utils/constant.h"

namespace Kernel {
// Keep the class template parameter list in one place so the declaration and
// out-of-line member definitions stay perfectly aligned.
#define QBMM_MX_KERNEL_NO_FULL_LOAD_CLASS_TEM_PARAMS \
    template <class ProblemShape, class BlockMmad, class BlockScheduler>
#define QBMM_MX_KERNEL_NO_FULL_LOAD_FUN_TEM_PARAMS ProblemShape, BlockMmad, BlockScheduler

using namespace AscendC;

QBMM_MX_KERNEL_NO_FULL_LOAD_CLASS_TEM_PARAMS
class QuantMatmulMxKernelSwat {
public:
    __aicore__ inline QuantMatmulMxKernelSwat()
    {}
    __aicore__ inline ~QuantMatmulMxKernelSwat()
    {}

    static constexpr bool weightNz = BlockMmad::weightNz;
    static constexpr bool transA = BlockMmad::transA;
    static constexpr bool transB = BlockMmad::transB;
    using AType = typename BlockMmad::AType;
    using BType = typename BlockMmad::BType;
    using ScaleAType = typename BlockMmad::ScaleAType;
    using ScaleBType = typename BlockMmad::ScaleBType;
    using CType = typename BlockMmad::CType;

    // The kernel layer does not own a scheduling policy itself; it selects the
    // concrete scheduler from the block pipeline traits.
    using BlockSchedulerOp =
        typename Block::BlockSchedulerSelector<ProblemShape, BlockScheduler, transA, transB, AType>::SchedulerOp;

    using BlockMmadParams = typename BlockMmad::Params;
    using L1Params = typename BlockMmad::L1Params;

    using TupleShape = AscendC::Shape<int64_t, int64_t, int64_t>;
    using BlockShape = AscendC::Shape<int64_t, int64_t, int64_t, int64_t>;
    using BlockCoord = AscendC::Coord<int64_t, int64_t, int64_t, int64_t>;
    using BlockSchedulerParams = typename BlockSchedulerOp::Params;

    using MakeLayoutA = typename BlockMmad::LayoutA;
    using MakeLayoutB = typename BlockMmad::LayoutB;
    using MakeLayoutScaleA = typename BlockMmad::LayoutScaleA;
    using MakeLayoutScaleB = typename BlockMmad::LayoutScaleB;

    struct QBMMTiling {
        // Base tile shape and L0C buffering mode selected by host tiling.
        uint32_t baseM;
        uint32_t baseN;
        uint32_t baseK;
        uint8_t dbL0C;
    };

    struct Params {
        // Translate the serialized tiling payload once at the launcher
        // boundary, then keep the kernel, scheduler, and block MMAD layers on
        // their own typed parameter bundles.
        ProblemShape problemShape;
        BlockMmadParams mmadParams;
        L1Params l1Params;
        BlockSchedulerParams schParams;
        QBMMTiling qbmmParams;
    };

public:
    // Launch one SWAT kernel instance on the current AIC block.
    __aicore__ inline void operator()(const Params& params);

private:
    // Bind raw GM addresses to typed kernel pointers once per launch.
    __aicore__ inline void ResetGmAddr(const Params& params);

    // Wrap the full GM tensors and repeatedly slice them into per-block views
    // according to the scheduler output.
    __aicore__ inline void Process(const Params& params, BlockSchedulerOp& bs);

    // `ProblemShape` stays generic at the template boundary, but the block
    // MMAD operator consumes a fixed 3D tuple internally.
    __aicore__ inline TupleShape ToShapeTuple(const ProblemShape& problemShape)
    {
        return {problemShape.m, problemShape.n, problemShape.k};
    }

private:
    BlockMmad mmadOp_;
    __gm__ AType* aGmAddr_;
    __gm__ BType* bGmAddr_;
    __gm__ CType* cGmAddr_;
    __gm__ ScaleAType* scaleAGmAddr_;
    __gm__ ScaleBType* scaleBGmAddr_;
};

QBMM_MX_KERNEL_NO_FULL_LOAD_CLASS_TEM_PARAMS
__aicore__ inline void QuantMatmulMxKernelSwat<QBMM_MX_KERNEL_NO_FULL_LOAD_FUN_TEM_PARAMS>::operator()(
    const Params& params)
{
    // TODO: AIV 核直接返回；AIC 核绑定 GM 地址、创建 BlockScheduler、初始化 BlockMmad 并调用 Process
    (void)params;
}

QBMM_MX_KERNEL_NO_FULL_LOAD_CLASS_TEM_PARAMS
__aicore__ inline void QuantMatmulMxKernelSwat<QBMM_MX_KERNEL_NO_FULL_LOAD_FUN_TEM_PARAMS>::ResetGmAddr(
    const Params& params)
{
    // TODO: 将 params.mmadParams 中的 GM 地址转换为 typed 指针并赋值给成员变量
    (void)params;
}

QBMM_MX_KERNEL_NO_FULL_LOAD_CLASS_TEM_PARAMS
__aicore__ inline void QuantMatmulMxKernelSwat<QBMM_MX_KERNEL_NO_FULL_LOAD_FUN_TEM_PARAMS>::Process(
    const Params& params, BlockSchedulerOp& bs)
{
    // Build full-GM tensor views once, then slice them per scheduled block.
    int64_t kScaleSize =
        ::CeilDiv(static_cast<int64_t>(params.problemShape.k), static_cast<int64_t>(MXFP_DIVISOR_SIZE)) *
        MXFP_MULTI_BASE_SIZE;
    auto layoutA = MakeLayoutA{}(params.problemShape.m, params.problemShape.k);
    auto layoutScaleA = MakeLayoutScaleA{}(params.problemShape.m, kScaleSize);
    auto layoutB = MakeLayoutB{}(params.problemShape.k, params.problemShape.n);
    auto layoutScaleB = MakeLayoutScaleB{}(kScaleSize, params.problemShape.n);
    auto layoutC =
        AscendC::Te::MakeFrameLayout<AscendC::Te::NDExtLayoutPtn>(params.problemShape.m, params.problemShape.n);

    auto gmA = AscendC::Te::MakeTensor(AscendC::Te::MakeMemPtr<AscendC::Te::Location::GM>(aGmAddr_), layoutA);
    auto gmScaleA =
        AscendC::Te::MakeTensor(AscendC::Te::MakeMemPtr<AscendC::Te::Location::GM>(scaleAGmAddr_), layoutScaleA);
    auto gmB = AscendC::Te::MakeTensor(AscendC::Te::MakeMemPtr<AscendC::Te::Location::GM>(bGmAddr_), layoutB);
    auto gmScaleB =
        AscendC::Te::MakeTensor(AscendC::Te::MakeMemPtr<AscendC::Te::Location::GM>(scaleBGmAddr_), layoutScaleB);
    auto gmC = AscendC::Te::MakeTensor(AscendC::Te::MakeMemPtr<AscendC::Te::Location::GM>(cGmAddr_), layoutC);

    BlockCoord blockIdx;
    constexpr int64_t kPos = 0L;
    while (bs.template GetTileIdx<weightNz>(blockIdx)) {
        // The scheduler packs GM origin into M/N and retains logical tile
        // indices in K/B so shape reconstruction still works.
        int64_t mPos = AscendC::Te::Get<MNK_M>(blockIdx);
        int64_t nPos = AscendC::Te::Get<MNK_N>(blockIdx);
        BlockShape singleShape = bs.template GetBlockShape<weightNz>(blockIdx);
        if (AscendC::Te::Get<MNK_M>(singleShape) <= 0 || AscendC::Te::Get<MNK_N>(singleShape) <= 0) {
            // Tail splitting can create empty logical slices; ignore them and
            // stop the current block once no useful work remains.
            return;
        }

        // `blockIdx` now carries both GM origin and logical tile metadata.
        auto gmBlockA =
            gmA.Slice(AscendC::Te::MakeCoord(mPos, kPos), AscendC::Te::MakeShape(AscendC::Te::Get<MNK_M>(singleShape), params.problemShape.k));
        auto gmBlockScaleA =
            gmScaleA.Slice(AscendC::Te::MakeCoord(mPos, kPos), AscendC::Te::MakeShape(AscendC::Te::Get<MNK_M>(singleShape), kScaleSize));
        auto gmBlockB =
            gmB.Slice(AscendC::Te::MakeCoord(kPos, nPos), AscendC::Te::MakeShape(params.problemShape.k, AscendC::Te::Get<MNK_N>(singleShape)));
        auto gmBlockScaleB =
            gmScaleB.Slice(AscendC::Te::MakeCoord(kPos, nPos), AscendC::Te::MakeShape(kScaleSize, AscendC::Te::Get<MNK_N>(singleShape)));
        auto gmBlockC =
            gmC.Slice(AscendC::Te::MakeCoord(mPos, nPos),
                AscendC::Te::MakeShape(AscendC::Te::Get<MNK_M>(singleShape), AscendC::Te::Get<MNK_N>(singleShape)));

        // TODO: 对当前 block 切片 GM Tensor，并调用 mmadOp_ 完成 L1/L0 MMAD 计算
        (void)gmBlockA;
        (void)gmBlockB;
        (void)gmBlockScaleA;
        (void)gmBlockScaleB;
        (void)gmBlockC;
        (void)singleShape;
    }
}

} // namespace Kernel

#undef QBMM_MX_KERNEL_NO_FULL_LOAD_CLASS_TEM_PARAMS
#undef QBMM_MX_KERNEL_NO_FULL_LOAD_FUN_TEM_PARAMS


---

## 步骤 3：Block 代码补齐

继续修改以下代码，完成 BlockMmad 内层循环中 MX MMAD 调用处标记 `TODO` 的部分。

In [ ]:
%%writefile src/01.07_blaze_based_matmul_operator_development/include/block/quant_matmul_mx_block_mmad_swat.h

/**
 * Copyright (c) 2026 Huawei Technologies Co., Ltd.
 * This program is free software, you can redistribute it and/or modify it under the terms and conditions of
 * CANN Open Software License Agreement Version 2.0 (the "License").
 * Please refer to the License for details. You may not use this file except in compliance with the License.
 * THIS SOFTWARE IS PROVIDED ON AN "AS IS" BASIS, WITHOUT WARRANTIES OF ANY KIND, EITHER EXPRESS OR IMPLIED,
 * INCLUDING BUT NOT LIMITED TO NON-INFRINGEMENT, MERCHANTABILITY, OR FITNESS FOR A PARTICULAR PURPOSE.
 * See LICENSE in the root of the software repository for the full text of the License.
 */

/*!
 * \file quant_matmul_mx_block_mmad_swat.h
 * \brief Block-level MX MMAD pipeline for the SWAT non-full-load path.
 */

#pragma once

#include "kernel_utils/common_utils.h"
#include "include/tensor_api/tensor.h"
#include "../policy/dispatch_policy.h"
#include "../utils/constant.h"
#include "../utils/layout_utils.h"
#include "../tile/tile_mmad_mx.h"
#include "../tile/copy_scale_l1_to_l0a.h"
#include "../tile/copy_scale_l1_to_l0b.h"
#include "../tile/pad_mx_kl1.h"

namespace Block {
using namespace AscendC;

template <
    class DispatchPolicy_, class ATypeTuple_, class LayoutATuple_, class BTypeTuple_, class LayoutBTuple_, class CType_,
    class LayoutC_>
class BlockMmad<
    DispatchPolicy_, ATypeTuple_, LayoutATuple_, BTypeTuple_, LayoutBTuple_, CType_, LayoutC_,
    AscendC::Std::enable_if_t<
        AscendC::Std::is_same_v<DispatchPolicy_, QuantMatmulMxMultiBlockWithSwat<NO_FULL_LOAD_MODE, 2UL>>>> {
public:
    template <typename T>
    struct TypeUnpack {
        using Data = T;
        using Scale = void;
    };

    template <typename T0, typename T1>
    struct TypeUnpack<AscendC::Std::tuple<T0, T1>> {
        using Data = T0;
        using Scale = T1;
    };

    template <typename T>
    struct LayoutUnpack {
        using Data = T;
        using Scale = void;
    };

    template <typename T0, typename T1>
    struct LayoutUnpack<AscendC::Std::tuple<T0, T1>> {
        using Data = T0;
        using Scale = T1;
    };

    using AType = typename TypeUnpack<ATypeTuple_>::Data;
    using ScaleAType = typename TypeUnpack<ATypeTuple_>::Scale;
    using BType = typename TypeUnpack<BTypeTuple_>::Data;
    using ScaleBType = typename TypeUnpack<BTypeTuple_>::Scale;
    using CType = CType_;
    using LayoutA = typename LayoutUnpack<LayoutATuple_>::Data;
    using LayoutScaleA = typename LayoutUnpack<LayoutATuple_>::Scale;
    using LayoutB = typename LayoutUnpack<LayoutBTuple_>::Data;
    using LayoutScaleB = typename LayoutUnpack<LayoutBTuple_>::Scale;
    using LayoutC = LayoutC_;
    using DispatchPolicy = DispatchPolicy_;
    using TupleShape = AscendC::Shape<int64_t, int64_t, int64_t>;
    using BlockShape = AscendC::Shape<int64_t, int64_t, int64_t, int64_t>;
    static constexpr bool weightNz = MatmulRecipe::IsWeightNz<LayoutB>::value;
    static constexpr bool transA = MatmulRecipe::IsTrans<LayoutA>::value;
    static constexpr bool transB = MatmulRecipe::IsTrans<LayoutB>::value;
    static constexpr bool isDTypeFp4 =
        AscendC::IsSameType<AType, fp4x2_e1m2_t>::value || AscendC::IsSameType<AType, fp4x2_e2m1_t>::value;
    static constexpr uint64_t L1_BUFFER_NUM = DispatchPolicy::stages;
    static constexpr uint64_t L1_BUFFER_MASK = L1_BUFFER_NUM - 1;
    static constexpr uint64_t L1_BUFFER_GROUP_NUM = L1_BUFFER_NUM >> 1;
    static constexpr uint64_t HALF_L0_SIZE = L0A_SIZE / DOUBLE_BUFFER_COUNT;
    static constexpr uint64_t HALF_L0C_SIZE = L0C_SIZE / DOUBLE_BUFFER_COUNT;
    static constexpr int32_t C0_SIZE = AscendC::AuxGetC0Size<AType>();
    static constexpr int32_t SCALE_C0 = 2;
    static constexpr int32_t L0C_C0 = 16;
    static constexpr uint64_t BLOCK_CUBE = 16UL;
    static constexpr uint64_t MXFP_GROUP_SIZE = 32UL;
    static constexpr uint64_t MXFP_DIVISOR_SIZE = 64UL;
    static constexpr uint64_t MXFP_MULTI_BASE_SIZE = 2UL;
    static constexpr uint64_t SCALE_BUFFER_NUM = 2;
    uint64_t m_{0UL};
    uint64_t n_{0UL};
    uint64_t k_{0UL};
    uint64_t kL1Iter_{0UL};
    uint64_t kL1_{0UL};
    uint64_t scaleKL1_{0UL};
    uint64_t baseM_{0UL};
    uint64_t baseN_{0UL};
    uint64_t baseK_{0UL};
    uint64_t abL1LoopCnt_{0UL};
    uint64_t scaleLoopCnt_{0UL};
    uint64_t l0PingPong_{0UL};
    uint64_t l0cPingPong_{0UL};
    bool enableL0cPingPong_{false};

    using MakeLayoutAL1 = AscendC::Te::FrameLayoutFormat<
        AscendC::Std::conditional_t<transA, AscendC::Te::ZNLayoutPtn, AscendC::Te::NZLayoutPtn>,
        AscendC::Std::Int<C0_SIZE>>;
    using MakeLayoutBL1 = AscendC::Te::FrameLayoutFormat<
        AscendC::Std::conditional_t<transB, AscendC::Te::ZNLayoutPtn, AscendC::Te::NZLayoutPtn>,
        AscendC::Std::Int<C0_SIZE>>;

    struct Params {
        GM_ADDR aGmAddr{nullptr};
        GM_ADDR bGmAddr{nullptr};
        GM_ADDR cGmAddr{nullptr};
        GM_ADDR scaleAGmAddr{nullptr};
        GM_ADDR scaleBGmAddr{nullptr};
    };

    struct L1Params {
        uint64_t kL1;
        uint64_t scaleKL1;
    };

    __aicore__ inline BlockMmad()
    {
// Prime all producer/consumer events so the first iteration can enter
// the pipelined copy-and-compute loop without special-case branches.
#pragma unroll
        for (uint8_t i = 0; i < MTE1_MTE2_EVENT_ID_NUM_MX; ++i) {
            AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>(i);
        }
        AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(ZERO_FLAG);
        AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(FIRST_FLAG);
        AscendC::SetMMLayoutTransform(true);
    }

    __aicore__ inline ~BlockMmad()
    {
// Drain every in-flight transfer before leaving so later blocks do not
// observe stale event state from the previous pipeline instance.
#pragma unroll
        for (uint8_t i = 0; i < MTE1_MTE2_EVENT_ID_NUM_MX; ++i) {
            AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(i);
        }
        AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(ZERO_FLAG);
        AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(FIRST_FLAG);
        AscendC::SetMMLayoutTransform(false);
    }

public:
    __aicore__ inline void Init(
        const TupleShape& problemShape, const BlockShape& l0TileShape, const L1Params& l1Params, bool enableL0cPingPong)
    {
        // Pre-compute all persistent buffer sizes and L1 offsets once per block
        // so the hot path only needs to switch between ping-pong slots.
        m_ = AscendC::Te::Get<IDX_M_IDX>(problemShape);
        n_ = AscendC::Te::Get<IDX_N_IDX>(problemShape);
        k_ = AscendC::Te::Get<IDX_K_IDX>(problemShape);
        kL1_ = l1Params.kL1;
        scaleKL1_ = l1Params.scaleKL1;
        baseM_ = AscendC::Te::Get<IDX_M_IDX>(l0TileShape);
        baseN_ = AscendC::Te::Get<IDX_N_IDX>(l0TileShape);
        baseK_ = AscendC::Te::Get<IDX_K_IDX>(l0TileShape);
        enableL0cPingPong_ = enableL0cPingPong;
        constexpr uint64_t sizeShift = isDTypeFp4 ? 1UL : 0UL;
        bL1OneBuffer_ = (baseN_ * kL1_) >> sizeShift;
        scaleBL1OneBuffer_ = baseN_ * CeilDiv(scaleKL1_, MXFP_DIVISOR_SIZE) * MXFP_MULTI_BASE_SIZE;
        aL1OneBuffer_ = (baseM_ * Align(kL1_, MXFP_DIVISOR_SIZE)) >> sizeShift;
        scaleAL1OneBuffer_ = baseM_ * CeilDiv(scaleKL1_, MXFP_DIVISOR_SIZE) * MXFP_MULTI_BASE_SIZE;
        scaleL1Window_ = scaleKL1_ / kL1_;
        kL1ScaleSize_ = CeilDiv(kL1_, MXFP_DIVISOR_SIZE) * MXFP_MULTI_BASE_SIZE;
        scaleKL1Group_ = CeilDiv(scaleKL1_, MXFP_GROUP_SIZE);
        scaleKL1ScaleSize_ = CeilDiv(scaleKL1_, MXFP_DIVISOR_SIZE) * MXFP_MULTI_BASE_SIZE;
        // 2 buffer: L1 space is : A0|B0|AScale0|BScale0|...|A1|B1|AScale1|BScale1|...
        // 4 buffer: L1 space is : A0A2|B0B2|AScale0|BScale0|...|A1A3|B1B3|AScale1|BScale1|...
        uint64_t l1HalfSize = AscendC::TOTAL_L1_SIZE >> 1;
#pragma unroll
        for (uint64_t bufferId = 0; bufferId < L1_BUFFER_NUM; ++bufferId) {
            uint64_t l1BufferGroup = bufferId >> 1;
            uint64_t l1HalfOffset = (bufferId & 1UL) * l1HalfSize;
            l1BufferAOffset_[bufferId] = l1HalfOffset + l1BufferGroup * aL1OneBuffer_;
            l1BufferBOffset_[bufferId] =
                l1HalfOffset + L1_BUFFER_GROUP_NUM * aL1OneBuffer_ + l1BufferGroup * bL1OneBuffer_;
        }
#pragma unroll
        for (int32_t bufferId = 0; bufferId < SCALE_BUFFER_NUM; bufferId++) {
            l1BufferScaleAOffset_[bufferId] = l1BufferBOffset_[bufferId] + bL1OneBuffer_ * L1_BUFFER_GROUP_NUM;
            l1BufferScaleBOffset_[bufferId] = l1BufferScaleAOffset_[bufferId] + scaleAL1OneBuffer_;
        }
        kL1Iter_ = CeilDiv(k_, kL1_);
    }

    template <typename TensorA, typename TensorB, typename TensorScaleA, typename TensorScaleB, typename TensorC>
    __aicore__ inline void operator()(
        TensorA gmA, TensorB gmB, TensorScaleA gmScaleA, TensorScaleB gmScaleB, TensorC gmC, BlockShape singleShape)
    {
        // Non-full-load streams both A and B through L1/L0 in chunks. Scale
        // tensors advance in a coarser cadence that matches `scaleKL1_`.
        auto curM = AscendC::Te::Get<IDX_M_TILEIDX>(singleShape);
        auto curN = AscendC::Te::Get<IDX_N_TILEIDX>(singleShape);
        uint64_t l0cOffset = (l0cPingPong_ & 1) * HALF_L0C_SIZE;
        auto layoutL0C = AscendC::Te::MakeFrameLayout<AscendC::Te::NZLayoutPtn, AscendC::Std::Int<L0C_C0>>(curM, curN);
        auto tensorL0C =
            AscendC::Te::MakeTensor(AscendC::Te::MakeMemPtr<AscendC::Te::Location::L0C, float>(l0cOffset), layoutL0C);
        uint64_t scaleWindowIter = 0;
        for (uint64_t iter0 = 0; iter0 < kL1Iter_; ++iter0) {
            uint64_t l1BufId = abL1LoopCnt_ & L1_BUFFER_MASK;
            uint64_t scaleL1BufId = scaleLoopCnt_ & 1;
            uint64_t kL1Offset = iter0 * kL1_;
            auto curGmBKL1 = (iter0 + 1 == kL1Iter_) ? (k_ - kL1Offset) : kL1_;
            auto curPadKL1 = CeilAlign(curGmBKL1, MXFP_DIVISOR_SIZE);
            auto curGmAKL1 = curGmBKL1;
            if (scaleWindowIter == 0) {
                // Scale fragments are refreshed only when the current K chunk
                // enters a new scale reuse window.
                AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(SCALE_BUFFER_FLAG_0 + scaleL1BufId);

                uint64_t curScaleKL1 = scaleKL1_;
                if (kL1Offset + curScaleKL1 > k_) {
                    curScaleKL1 = k_ - kL1Offset;
                }

                auto CopyScaleGM2L1 = AscendC::Te::MakeCopy(AscendC::Te::CopyGM2L1{});
                auto layoutScaleAL1 =
                    AscendC::Te::MakeFrameLayout<AscendC::Te::ZZLayoutPtn, AscendC::Std::Int<SCALE_C0>>(
                        curM, scaleKL1Group_);
                auto tensorScaleAL1 = AscendC::Te::MakeTensor(
                    AscendC::Te::MakeMemPtr<AscendC::Te::Location::L1, fp8_e8m0_t>(l1BufferScaleAOffset_[scaleL1BufId]),
                    layoutScaleAL1);
                auto gmBlockScaleA = gmScaleA.Slice(
                    AscendC::Te::MakeCoord(0, kL1Offset / MXFP_GROUP_SIZE),
                    AscendC::Te::MakeShape(curM, CeilDiv(curScaleKL1, MXFP_DIVISOR_SIZE) * MXFP_MULTI_BASE_SIZE));
                AscendC::Te::Copy(CopyScaleGM2L1, tensorScaleAL1, gmBlockScaleA);

                auto layoutScaleBL1 =
                    AscendC::Te::MakeFrameLayout<AscendC::Te::NNLayoutPtn, AscendC::Std::Int<SCALE_C0>>(
                        scaleKL1Group_, curN);
                auto tensorScaleBL1 = AscendC::Te::MakeTensor(
                    AscendC::Te::MakeMemPtr<AscendC::Te::Location::L1, fp8_e8m0_t>(l1BufferScaleBOffset_[scaleL1BufId]),
                    layoutScaleBL1);
                auto gmBlockScaleB = gmScaleB.Slice(
                    AscendC::Te::MakeCoord(kL1Offset / MXFP_GROUP_SIZE, 0),
                    AscendC::Te::MakeShape(CeilDiv(curScaleKL1, MXFP_DIVISOR_SIZE) * MXFP_MULTI_BASE_SIZE, curN));
                AscendC::Te::Copy(CopyScaleGM2L1, tensorScaleBL1, gmBlockScaleB);
            }

            AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(l1BufId);
            auto copyGM2L1 = AscendC::Te::MakeCopy(AscendC::Te::CopyGM2L1{});
            auto layoutAL1 = MakeLayoutAL1{}(curM, curPadKL1);
            auto tensorAL1 = AscendC::Te::MakeTensor(
                AscendC::Te::MakeMemPtr<AscendC::Te::Location::L1, AType>(l1BufferAOffset_[l1BufId]), layoutAL1);
            auto gmBlockA = gmA.Slice(AscendC::Te::MakeCoord(0, kL1Offset), AscendC::Te::MakeShape(curM, curGmAKL1));
            ::Tile::PadMxKAL1::PadZero(tensorAL1, gmBlockA);
            AscendC::Te::Copy(copyGM2L1, tensorAL1, gmBlockA);

            auto layoutBL1 = MakeLayoutBL1{}(curPadKL1, curN);
            auto tensorBL1 = AscendC::Te::MakeTensor(
                AscendC::Te::MakeMemPtr<AscendC::Te::Location::L1, BType>(l1BufferBOffset_[l1BufId]), layoutBL1);
            auto gmBlockB = gmB.Slice(AscendC::Te::MakeCoord(kL1Offset, 0), AscendC::Te::MakeShape(curGmBKL1, curN));
            ::Tile::PadMxKBL1::PadZero(tensorBL1, gmBlockB);
            AscendC::Te::Copy(copyGM2L1, tensorBL1, gmBlockB);

            AscendC::SetFlag<AscendC::HardEvent::MTE2_MTE1>(l1BufId);
            AscendC::WaitFlag<AscendC::HardEvent::MTE2_MTE1>(l1BufId);

            uint64_t kL0Iter = CeilDiv(curGmBKL1, baseK_);
            for (uint16_t iter1 = 0; iter1 < kL0Iter; ++iter1) {
                // Each inner iteration slices the current L1 chunk into one
                // L0-sized MMAD tile and accumulates it into L0C.
                auto kL0Offset = iter1 * baseK_;
                auto curKL0 = (kL0Offset + baseK_ > curPadKL1) ? (curPadKL1 - kL0Offset) : baseK_;
                uint64_t l0BufId = l0PingPong_ & 0x1;
                uint64_t l0Offset = HALF_L0_SIZE * l0BufId;
                AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(l0BufId);

                auto CopyL12L0A = AscendC::Te::MakeCopy(AscendC::Te::CopyL12L0A{});
                auto CopyL12L0B = AscendC::Te::MakeCopy(AscendC::Te::CopyL12L0B{});
                auto layoutAL0 =
                    AscendC::Te::MakeFrameLayout<AscendC::Te::NZLayoutPtn, AscendC::Std::Int<C0_SIZE>>(curM, curKL0);
                auto tensorAL0 = AscendC::Te::MakeTensor(
                    AscendC::Te::MakeMemPtr<AscendC::Te::Location::L0A, AType>(l0Offset), layoutAL0);
                auto tensorBlockAL1 =
                    tensorAL1.Slice(AscendC::Te::MakeCoord(0, kL0Offset), AscendC::Te::MakeShape(curM, curKL0));
                AscendC::Te::Copy(CopyL12L0A, tensorAL0, tensorBlockAL1);

                auto layoutBL0 =
                    AscendC::Te::MakeFrameLayout<AscendC::Te::ZNLayoutPtn, AscendC::Std::Int<C0_SIZE>>(curKL0, curN);
                auto tensorBL0 = AscendC::Te::MakeTensor(
                    AscendC::Te::MakeMemPtr<AscendC::Te::Location::L0B, BType>(l0Offset), layoutBL0);
                auto tensorBlockBL1 =
                    tensorBL1.Slice(AscendC::Te::MakeCoord(kL0Offset, 0), AscendC::Te::MakeShape(curKL0, curN));
                AscendC::Te::Copy(CopyL12L0B, tensorBL0, tensorBlockBL1);

                auto coordScaleKL1 = scaleWindowIter * kL1ScaleSize_;
                auto layoutScaleAL0 =
                    AscendC::Te::MakeFrameLayout<AscendC::Te::ZZLayoutPtn, AscendC::Std::Int<SCALE_C0>>(
                        curM, CeilDiv(curKL0, MXFP_DIVISOR_SIZE) * MXFP_MULTI_BASE_SIZE);
                auto tensorScaleAL0 = AscendC::Te::MakeTensor(
                    AscendC::Te::MakeMemPtr<AscendC::Te::Location::L0A, fp8_e8m0_t>(l0Offset), layoutScaleAL0);
                auto layoutScaleAL1 =
                    AscendC::Te::MakeFrameLayout<AscendC::Te::ZZLayoutPtn, AscendC::Std::Int<SCALE_C0>>(
                        curM, scaleKL1ScaleSize_);
                auto tensorScaleAL1 = AscendC::Te::MakeTensor(
                    AscendC::Te::MakeMemPtr<AscendC::Te::Location::L1, fp8_e8m0_t>(l1BufferScaleAOffset_[scaleL1BufId]),
                    layoutScaleAL1);
                auto tensorBlockScaleAL1 = tensorScaleAL1.Slice(
                    AscendC::Te::MakeCoord(0, coordScaleKL1), AscendC::Te::MakeShape(curM, kL1ScaleSize_));
                auto CopyL12L0MxScaleA3510 = AscendC::Te::MakeCopy(::Tile::CopyL12L0MxScaleA3510{});
                CopyL12L0MxScaleA3510.Call(
                    tensorScaleAL0, tensorBlockScaleAL1, AscendC::Te::MakeCoord(0, kL0Offset));

                auto layoutScaleBL0 =
                    AscendC::Te::MakeFrameLayout<AscendC::Te::NNLayoutPtn, AscendC::Std::Int<SCALE_C0>>(
                        CeilDiv(curKL0, MXFP_DIVISOR_SIZE) * MXFP_MULTI_BASE_SIZE, curN);
                auto tensorScaleBL0 = AscendC::Te::MakeTensor(
                    AscendC::Te::MakeMemPtr<AscendC::Te::Location::L0B, fp8_e8m0_t>(l0Offset), layoutScaleBL0);
                auto layoutScaleBL1 =
                    AscendC::Te::MakeFrameLayout<AscendC::Te::NNLayoutPtn, AscendC::Std::Int<SCALE_C0>>(
                        scaleKL1ScaleSize_, curN);
                auto tensorScaleBL1 = AscendC::Te::MakeTensor(
                    AscendC::Te::MakeMemPtr<AscendC::Te::Location::L1, fp8_e8m0_t>(l1BufferScaleBOffset_[scaleL1BufId]),
                    layoutScaleBL1);
                auto tensorBlockScaleBL1 = tensorScaleBL1.Slice(
                    AscendC::Te::MakeCoord(coordScaleKL1, 0), AscendC::Te::MakeShape(kL1ScaleSize_, curN));
                auto CopyL12L0MxScaleB3510 = AscendC::Te::MakeCopy(::Tile::CopyL12L0MxScaleB3510{});
                CopyL12L0MxScaleB3510.Call(
                    tensorScaleBL0, tensorBlockScaleBL1, AscendC::Te::MakeCoord(kL0Offset, 0));

                AscendC::SetFlag<AscendC::HardEvent::MTE1_M>(l0BufId);
                AscendC::WaitFlag<AscendC::HardEvent::MTE1_M>(l0BufId);
                uint8_t mmadUnitFlag =
                    (iter0 + 1 == kL1Iter_ && iter1 + 1 == kL0Iter) ? FINAL_ACCUMULATION : NON_FINAL_ACCUMULATION;
                bool mmadCmatrixInitVal = (iter0 == 0 && iter1 == 0);
                AscendC::Te::MmadParams mmadParams;
                // TODO: 配置 mmadParams.m/k/n/unitFlag/cmatrixInitVal，并调用 AscendC::Te::Mmad 完成 MX 量化矩阵乘累加
                // 提示: 使用 MmadTraits<MmadOperation, MmadTraitMX>，K 维需 CeilAlign(curKL0, MXFP_DIVISOR_SIZE)
                (void)mmadUnitFlag;
                (void)mmadCmatrixInitVal;
                (void)mmadParams;
                (void)tensorL0C;
                (void)tensorAL0;
                (void)tensorBL0;
                AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(l0BufId);
                l0PingPong_++;
            }

            // Release the current L1 slot only after every L0 slice derived
            // from it has completed its MMAD accumulation.
            AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>(l1BufId);
            if (scaleWindowIter + 1 == scaleL1Window_ || iter0 == kL1Iter_ - 1) {
                AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>(SCALE_BUFFER_FLAG_0 + scaleL1BufId);
                scaleLoopCnt_++;
                scaleWindowIter = 0;
            } else {
                ++scaleWindowIter;
            }
            abL1LoopCnt_++;
        }

        auto CopyL0C2GM = AscendC::Te::MakeCopy(AscendC::Te::CopyL0C2GM{});
        // The whole block accumulates into one L0C tile, which is flushed once
        // after all K chunks have contributed.
        AscendC::Te::Copy(
            CopyL0C2GM.with(AscendC::Te::FixpipeParams{FINAL_ACCUMULATION}), gmC, tensorL0C);
        if (enableL0cPingPong_) {
            l0cPingPong_++;
        }
    }

private:
    uint64_t aL1OneBuffer_ = 0UL;
    uint64_t bL1OneBuffer_ = 0UL;
    uint64_t scaleAL1OneBuffer_ = 0UL;
    uint64_t scaleBL1OneBuffer_ = 0UL;
    uint64_t scaleL1Window_ = 0UL;
    uint64_t kL1ScaleSize_ = 0UL;
    uint64_t scaleKL1Group_ = 0UL;
    uint64_t scaleKL1ScaleSize_ = 0UL;
    uint64_t l1BufferAOffset_[L1_BUFFER_NUM] = {0UL};
    uint64_t l1BufferBOffset_[L1_BUFFER_NUM] = {0UL};
    uint64_t l1BufferScaleAOffset_[SCALE_BUFFER_NUM] = {0UL};
    uint64_t l1BufferScaleBOffset_[SCALE_BUFFER_NUM] = {0UL};
};
} // namespace Block


---

## 步骤 4：部署算子运行测试

Kernel 与 Block 代码补齐完成后，执行如下命令编译算子并运行端到端测试。测试参数：**m=16, k=128, n=16384, transA=0, transB=1**。

In [ ]:
!bash src/01.07_blaze_based_matmul_operator_development/build.sh

In [ ]:
%%bash
cd src/01.07_blaze_based_matmul_operator_development/build_out
python3 gen_data.py 16 128 16384 0 1
./quant_matmul_mxfp8_tutorial 16 128 16384 0 1

运行成功后应看到：

```
[PASS] NPU results are consistent with CPU.
```

### 执行以下代码获取 Kernel / Block 参考答案。

**Kernel 参考答案**

In [ ]:
!cat answer/01.07_answer/include/kernel/quant_matmul_mx_kernel_swat.h

**Block 参考答案**

In [ ]:
!cat answer/01.07_answer/include/block/quant_matmul_mx_block_mmad_swat.h